# Challenge: Build a Parking Gate

Automatic parking gates like the ones common in Mongolia use a camera to read a car's license plate. If the plate is recognized, the gate lifts; if not, it stays down. In this notebook you'll build your own version: a VLM will read a license plate you hold up to the camera, and your code decides whether the gate opens.

<a target="_blank" href="https://codetto.app?github=simonguest/codercub/blob/main/labs/09/notebooks/parking_gate_challenge.ipynb">
  <img src="https://img.shields.io/badge/Open_in-Codetto-blue" alt="Open In Codetto"/>
</a>

## The Challenge

Write your license plate number on a piece of paper. Then write code that:

1. Captures a photo of your plate from the webcam
2. Sends it to a VLM and asks it to read the text on the plate
3. Checks whether the text it reads matches the expected plate
4. Opens the gate if it matches, and keeps it closed if it doesn't

The gate graphics and the `open_gate()` / `close_gate()` functions are provided for you below — your job is the camera capture and the VLM call, which you've already done in `vlm.ipynb` and `vlm_reasoning.ipynb`.

In [ ]:
from openai import OpenAI
import os
import math
import time

client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=os.environ["OPENROUTER_API_KEY"],
)

VLM_MODEL = "qwen/qwen3-vl-8b-instruct"

CLOSED_COLOR = "#e94560"
OPEN_COLOR = "#44cc88"
MOVING_COLOR = "#f5a623"

def draw_gate(ctx, w, h, angle, label, color):
  """Draws the barrier arm at the given angle (0 = closed/horizontal, 85 = open/vertical) with a status label."""
  canvas.clear()

  post_x, post_y = 50, h - 40
  ctx.fill_style = "#222222"
  ctx.fill_rect(post_x - 8, post_y - 70, 16, 70)

  rad = math.radians(angle)
  length = 700
  end_x = post_x + length * math.cos(rad)
  end_y = (post_y - 70) - length * math.sin(rad)

  ctx.stroke_style = color
  ctx.line_width = 10
  ctx.begin_path()
  ctx.move_to(post_x, post_y - 70)
  ctx.line_to(end_x, end_y)
  ctx.stroke()

  ctx.fill_style = "rgba(0, 0, 0, 0.55)"
  ctx.fill_rect(w - 190, 10, 180, 40)
  ctx.fill_style = color
  ctx.font = "bold 20px sans-serif"
  ctx.text_align = "left"
  ctx.text_baseline = "middle"
  ctx.fill_text(label, w - 175, 30)

def open_gate(ctx, w, h):
  """Animates the barrier swinging up from closed to open."""
  for angle in range(0, 90, 15):
    draw_gate(ctx, w, h, angle, "GATE: OPENING", MOVING_COLOR)
    time.sleep(0.08)
  draw_gate(ctx, w, h, 85, "GATE: OPEN", OPEN_COLOR)

def close_gate(ctx, w, h):
  """Animates the barrier swinging back down to closed."""
  for angle in range(85, -1, -15):
    draw_gate(ctx, w, h, angle, "GATE: CLOSING", MOVING_COLOR)
    time.sleep(0.08)
  draw_gate(ctx, w, h, 0, "GATE: CLOSED", CLOSED_COLOR)

def display_countdown(message):
  ctx.fill_style = "rgba(0, 0, 0, 0.55)"
  ctx.fill_rect(10, 10, 200, 40)
  ctx.fill_style = "#ffffff"
  ctx.font = "bold 20px sans-serif"
  ctx.text_align = "left"
  ctx.text_baseline = "middle"
  ctx.fill_text(message, 15, 30)


## Your Challenge

Run the cell below to see the closed gate. Then fill in the four `TODO`s:

- **TODO 1**: Write the correct instruction for the VLM to read the plate from the scene
- **TODO 2**: Check whether `ALLOWED_PLATE` appears in the VLM's response. If it does, call `open_gate(ctx, w, h)`. If not, print `"Access denied"` and leave the gate closed.

In [ ]:
import cv
import graphics

ALLOWED_PLATE = "COD3R" #@param

canvas = graphics.canvas()
camera = cv.start_camera()
ctx = canvas.get_context('2d')
w = canvas.get_width()
h = canvas.get_height()

# Draw gate in closed position
draw_gate(ctx, w, h, 0, "GATE: CLOSED", CLOSED_COLOR)

# Display countdown
for count in ["3", "2", "1"]:
  display_countdown(f"Scanning plate in {count}")
  time.sleep(1)
display_countdown("Scanning...")

image_url = cv.capture_frame(camera)

# TODO 1: Fill in the instruction for the VLM

response = client.chat.completions.create(
  model=VLM_MODEL,
  messages=[{
    "role": "user",
    "content": [
      {"type": "image_url", "image_url": {"url": image_url}},
      {"type": "text", "text": "INSTRUCTION FOR THE VLM"},
    ]
  }],
  stream=False
)

vlm_response = response.choices[0].message.content.strip()
print(f"VLM read: {vlm_response}")

# TODO 2: If ALLOWED_PLATE appears in vlm_response, open the gate. Otherwise,
#         keep it closed and print a denied message.
if False:  # <-- replace this condition
  open_gate(ctx, w, h)
else:
  draw_gate(ctx, w, h, 0, "ACCESS DENIED", CLOSED_COLOR)
  print("Access denied - plate not recognized.")

# Stop the camera
camera.stop()


VLM read: Analyze the image and provide a detailed description of the scene, including the person’s appearance, actions, and the surrounding environment. Describe any visible emotions or body language that might indicate the person’s state of mind. If there are any objects or features in the background or foreground, describe them as well. Avoid making assumptions or adding information not directly visible in the image.

Access denied - plate not recognized.



Think about the full pipeline you just built: camera capture, VLM reading, and a plain Python string check.

Did the VLM read your plate correctly every time? What kinds of mistakes did it make, if any (extra spaces, mixed-up characters, lowercase vs. uppercase)? How would you make the matching check more forgiving of small misreads?

## Ideas to Extend This App

- **Auto re-close**: after `open_gate()`, wait a few seconds and call `close_gate()` automatically, like a real barrier resetting for the next car
- **Multiple allowed plates**: check `plate_text` against a list of allowed plates instead of just one
- **Log attempts**: keep a Python list of every plate read and whether it was let in, then print it at the end

## Check for Understanding